# Scenario Construction

## 1. Create Benign Files

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.Scenario import BenignFlowExtractor, WindowDatasetBuilder, WindowScenarioBuilder

### 1.1 Extract Benign Flows

In [ ]:
extractBenignFlows = False
benignExtractionDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
benignFlowsOutputPath = Path("data/benign/benign_flows.csv")
benignExtractionChunkSize = 100_000
benignSourceFiles = None
overwriteBenignFlows = False

if extractBenignFlows:
    benignExtraction = BenignFlowExtractor(
        dataDirectory=benignExtractionDirectory,
        chunkSize=benignExtractionChunkSize,
    ).extract(
        outputPath=benignFlowsOutputPath,
        sourceFiles=benignSourceFiles,
        overwrite=overwriteBenignFlows,
    )
    display(pd.DataFrame([benignExtraction]))

### 1.2 Aggregate Benign Flows

In [ ]:
benignFlowsPath = Path("data/benign/benign_flows.csv")
benignAggregationOutputDirectory = Path("data")
benignAggregationChunkSize = 100_000
benignWindowSeconds = [1.0]

benignDatasets = WindowDatasetBuilder(
    dataDirectory=benignFlowsPath.parent,
    outputDirectory=benignAggregationOutputDirectory,
    chunkSize=benignAggregationChunkSize,
).aggregateBenignFlows(
    benignFlowsPath=benignFlowsPath,
    windowSeconds=benignWindowSeconds,
)

display(benignDatasets["summary"])

### 1.3 Stratify Benign Flows

In [1]:
from pathlib import Path
from IPython.display import display
from src.Scenario.SequentialFlowScenarioBuilder import SequentialFlowScenarioBuilder

DataDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")

ExcludedFiles = [
    "DrDoS_DNS.csv",
    "DrDoS_LDAP.csv",
    "DrDoS_UDP.csv",
    "DrDoS_SSDP.csv",
    "Syn.csv",
]

FilteredBenignPath = Path("data/benign/benign_flows_filtered.csv")
BenignOutputPath = Path("data/benign/benign_flows_sampled_15000.csv")

BenignSampleCount = 15000
SamplingSeed = 42
ChunkSize = 100_000
Overwrite = True

benignResult = SequentialFlowScenarioBuilder.prepareBenignReference(
    dataDirectory=DataDirectory,
    excludedFiles=ExcludedFiles,
    filteredPath=FilteredBenignPath,
    outputPath=BenignOutputPath,
    sampleCount=BenignSampleCount,
    randomSeed=SamplingSeed,
    chunkSize=ChunkSize,
    overwrite=Overwrite,
)

display(benignResult)

,Dataset,Path,Instances,SamplingSeed,EligibleBenignInstances,SourceFiles
0,BENIGN,data\benign\benign_flows_sampled_15000.csv,15000,42,48537,6


## 2. Create Attack Files

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.Scenario import WindowDatasetBuilder, WindowScenarioBuilder

attackDataDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
attackAggregationOutputDirectory = Path("data")
attackAggregationChunkSize = 100_000
attackWindowSeconds = [1.0]
cutMinutes = None
cutStartMinutes = {}

attackFiles = {'NTP': 'DrDoS_NTP.csv',
    'DNS': 'DrDoS_DNS.csv',
    'LDAP': 'DrDoS_LDAP.csv',
    'MSSQL': 'DrDoS_MSSQL.csv',
    'NETBIOS': 'DrDoS_NetBIOS.csv',
    'SNMP': 'DrDoS_SNMP.csv',
    'SSDP': 'DrDoS_SSDP.csv',
    'UDP': 'DrDoS_UDP.csv',
    'UDP-LAG': 'UDPLag.csv',
    'SYN': 'Syn.csv',
    'TFTP': 'TFTP.csv'}

attackDatasets = WindowDatasetBuilder(
    dataDirectory=attackDataDirectory,
    outputDirectory=attackAggregationOutputDirectory,
    chunkSize=attackAggregationChunkSize,
).aggregateAttackFlows(
    attackFiles=attackFiles,
    windowSeconds=attackWindowSeconds,
    cutMinutes=cutMinutes,
    cutStartMinutes=cutStartMinutes,
)

display(attackDatasets["summary"])

## 3. Create Similar Dataset

In [ ]:
similarityOutputDirectory = Path("data/similarity")
similarityBenignPath = Path("data/benign/benign_1s.csv")
similarityAttacksDirectory = Path("data/attacks/Window_1s")
similaritySequence = ["NTP", "DNS", "LDAP", "MSSQL", "NETBIOS", "SNMP", "SSDP", "UDP", "UDP-LAG", "SYN", "TFTP"]
similarityBlockSeconds = 180
similarityInitialBenignSeconds = 2000
similarityBetweenBenignSeconds = 600
similarityAttackPositionsSeconds = None
similarityBenignStartSeconds = 0

similarityDataset = WindowScenarioBuilder(
    benignPath=similarityBenignPath,
    attacksDirectory=similarityAttacksDirectory,
    windowSeconds=1.0,
).build(
    sequence=similaritySequence,
    outputPath=similarityOutputDirectory / f"similarity_1s.csv",
    blockSeconds=similarityBlockSeconds,
    initialBenignSeconds=similarityInitialBenignSeconds,
    betweenBenignSeconds=similarityBetweenBenignSeconds,
    randomSeed=42,
    attackPositionsSeconds=similarityAttackPositionsSeconds,
    benignStartSeconds=similarityBenignStartSeconds,
    allowBenignReuse=True,
)

display(similarityDataset["summary"])

## 4. Create Controlled Window Scenarios

### 4.1. WINDOW - DNS-SYN-LDAP

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.Scenario import WindowScenarioBuilder

BenignPath = Path("data/benign/benign_1s.csv")
AttacksDirectory = Path("data/attacks/Window_1s")
OutputDirectory = Path("data/scenario/WIN_DNS-SYN-LDAP")

BlockSeconds = [60, 120, 238]
RepeatAttackBlocks = True
InitialBenignSamples = 3600
BetweenBenignSamples = 1200
RecurrenceBetweenBenignSamples = 600

Scenarios = {
    "Consistency": ['DNS', 'DNS', 'DNS'],
    "Generalization": ['DNS', 'LDAP', 'DNS'],
    "Adaptation": ['DNS', 'SYN', 'DNS'],
    "Recurrence": ['DNS', 'SYN', 'LDAP', 'DNS', 'SYN', 'LDAP'],
}

builder = WindowScenarioBuilder(
    benignPath=BenignPath,
    attacksDirectory=AttacksDirectory,
    windowSeconds=1.0,
)

results = []
for scenarioName, sequence in Scenarios.items():
    result = builder.buildVariants(
        repeatAttackBlocks=RepeatAttackBlocks,
        scenarioName=scenarioName, sequence=sequence,
        outputDirectory=OutputDirectory, blockSeconds=BlockSeconds,
        initialBenignSeconds=InitialBenignSamples,
        betweenBenignSeconds=(
            RecurrenceBetweenBenignSamples
            if scenarioName == "Recurrence" else BetweenBenignSamples
        ),
        randomSeed=42,
    )
    results.append(result["results"])

display(result["availability"])
display(pd.concat(results, ignore_index=True))

### 4.2. WINDOW - UDP-SYN-SSDP

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.Scenario import WindowScenarioBuilder

BenignPath = Path("data/benign/benign_1s.csv")
AttacksDirectory = Path("data/attacks/Window_1s")
OutputDirectory = Path("data/scenario/WIN_UDP-SYN-SSDP")

BlockSeconds = [60, 120, 238]
RepeatAttackBlocks = True
InitialBenignSamples = 3600
BetweenBenignSamples = 1200
RecurrenceBetweenBenignSamples = 600

Scenarios = {
    "Consistency": ['UDP', 'UDP', 'UDP'],
    "Generalization": ['UDP', 'SSDP', 'UDP'],
    "Adaptation": ['UDP', 'SYN', 'UDP'],
    "Recurrence": ['UDP', 'SYN', 'SSDP', 'UDP', 'SYN', 'SSDP'],
}

builder = WindowScenarioBuilder(
    benignPath=BenignPath,
    attacksDirectory=AttacksDirectory,
    windowSeconds=1.0,
)

results = []
for scenarioName, sequence in Scenarios.items():
    result = builder.buildVariants(
        repeatAttackBlocks=RepeatAttackBlocks,
        scenarioName=scenarioName, sequence=sequence,
        outputDirectory=OutputDirectory, blockSeconds=BlockSeconds,
        initialBenignSeconds=InitialBenignSamples,
        betweenBenignSeconds=(
            RecurrenceBetweenBenignSamples
            if scenarioName == "Recurrence" else BetweenBenignSamples
        ),
        randomSeed=42,
    )
    results.append(result["results"])

display(result["availability"])
display(pd.concat(results, ignore_index=True))

## 5. Create Controlled Flow Scenarios

### 5.1. FLOW - DNS-SYN-LDAP

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.Scenario.SequentialFlowScenarioBuilder import SequentialFlowScenarioBuilder

BenignPath = Path("data/benign/benign_flows_sampled_15000.csv")
AttacksDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
OutputDirectory = Path("data/scenario/FLOW-DNS-SYN-LDAP")

BlockInstances = [25, 200, 1000]
BenignSampleCount = 15000
RepeatAttackBlocks = False
InitialBenignSamples = 6000
BetweenBenignSamples = 1200
RecurrenceBetweenBenignSamples = 600
ChunkSize = 100_000

AttackFiles = {
    "DNS": "DrDoS_DNS.csv",
    "SYN": "Syn.csv",
    "LDAP": "DrDoS_LDAP.csv",
}

Scenarios = {
    "Consistency": ["DNS", "DNS", "DNS"],
    "Generalization": ["DNS", "LDAP", "DNS"],
    "Adaptation": ["DNS", "SYN", "DNS"],
    "Recurrence": ["DNS", "SYN", "LDAP", "DNS", "SYN", "LDAP"],
}

builder = SequentialFlowScenarioBuilder(
    benignPath=BenignPath,
    dataDirectory=AttacksDirectory,
    attackFiles=AttackFiles,
    chunkSize=ChunkSize,
)

results = []

for scenarioName, sequence in Scenarios.items():
    result = builder.buildVariants(
        scenarioName=scenarioName,
        sequence=sequence,
        outputDirectory=OutputDirectory,
        blockInstances=BlockInstances,
        benignSampleCount=BenignSampleCount,
        repeatAttackBlocks=RepeatAttackBlocks,
        initialBenignInstances=InitialBenignSamples,
        betweenBenignInstances=(
            RecurrenceBetweenBenignSamples
            if scenarioName == "Recurrence"
            else BetweenBenignSamples
        ),
    )
    results.append(result["results"])

display(result["availability"])
display(pd.concat(results, ignore_index=True))

,Attack,AvailableAttackFlows,OriginalStart,OriginalEnd
0,DNS,5071011,2018-12-01 10:51:39.820844,2018-12-01 11:22:40.254721
1,LDAP,2179930,2018-12-01 11:22:40.255362,2018-12-01 11:32:32.915439
2,SYN,1582289,2018-12-01 13:30:30.741566,2018-12-01 13:34:27.403304


,Scenario,Family,BlockInstances,RepeatAttackBlocks,Status,Reason,Path,BlocksPath,Instances,AttackInstances
0,Consistency,FLOW-DNS-SYN-LDAP,25,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Consistency\Co...,data\scenario\FLOW-DNS-SYN-LDAP\Consistency\Co...,15078,75
1,Consistency,FLOW-DNS-SYN-LDAP,200,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Consistency\Co...,data\scenario\FLOW-DNS-SYN-LDAP\Consistency\Co...,15839,600
2,Consistency,FLOW-DNS-SYN-LDAP,1000,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Consistency\Co...,data\scenario\FLOW-DNS-SYN-LDAP\Consistency\Co...,18514,3000
3,Generalization,FLOW-DNS-SYN-LDAP,25,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Generalization...,data\scenario\FLOW-DNS-SYN-LDAP\Generalization...,15077,75
4,Generalization,FLOW-DNS-SYN-LDAP,200,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Generalization...,data\scenario\FLOW-DNS-SYN-LDAP\Generalization...,15628,600
5,Generalization,FLOW-DNS-SYN-LDAP,1000,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Generalization...,data\scenario\FLOW-DNS-SYN-LDAP\Generalization...,18497,3000
6,Adaptation,FLOW-DNS-SYN-LDAP,25,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Adaptation\Ada...,data\scenario\FLOW-DNS-SYN-LDAP\Adaptation\Ada...,15079,75
7,Adaptation,FLOW-DNS-SYN-LDAP,200,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Adaptation\Ada...,data\scenario\FLOW-DNS-SYN-LDAP\Adaptation\Ada...,15630,600
8,Adaptation,FLOW-DNS-SYN-LDAP,1000,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Adaptation\Ada...,data\scenario\FLOW-DNS-SYN-LDAP\Adaptation\Ada...,18499,3000
9,Recurrence,FLOW-DNS-SYN-LDAP,25,False,Gerado,,data\scenario\FLOW-DNS-SYN-LDAP\Recurrence\Rec...,data\scenario\FLOW-DNS-SYN-LDAP\Recurrence\Rec...,15154,150


### 5.2. FLOW - UDP-SYN-SSDP

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.Scenario.SequentialFlowScenarioBuilder import SequentialFlowScenarioBuilder

BenignPath = Path("data/benign/benign_flows_sampled_15000.csv")
AttacksDirectory = Path(r"E:\CICDDoS2019\01-12\CSV")
OutputDirectory = Path("data/scenario/FLOW-UDP-SYN-SSDP")

BlockInstances = [25, 200, 1000]
BenignSampleCount = 15000
RepeatAttackBlocks = False
InitialBenignSamples = 6000
BetweenBenignSamples = 1200
RecurrenceBetweenBenignSamples = 600
ChunkSize = 100_000

AttackFiles = {
    "UDP": "DrDoS_UDP.csv",
    "SYN": "Syn.csv",
    "SSDP": "DrDoS_SSDP.csv",
}

Scenarios = {
    "Consistency": ["UDP", "UDP", "UDP"],
    "Generalization": ["UDP", "SSDP", "UDP"],
    "Adaptation": ["UDP", "SYN", "UDP"],
    "Recurrence": ["UDP", "SYN", "SSDP", "UDP", "SYN", "SSDP"],
}

builder = SequentialFlowScenarioBuilder(
    benignPath=BenignPath,
    dataDirectory=AttacksDirectory,
    attackFiles=AttackFiles,
    chunkSize=ChunkSize,
)

results = []

for scenarioName, sequence in Scenarios.items():
    result = builder.buildVariants(
        scenarioName=scenarioName,
        sequence=sequence,
        outputDirectory=OutputDirectory,
        blockInstances=BlockInstances,
        benignSampleCount=BenignSampleCount,
        repeatAttackBlocks=RepeatAttackBlocks,
        initialBenignInstances=InitialBenignSamples,
        betweenBenignInstances=(
            RecurrenceBetweenBenignSamples
            if scenarioName == "Recurrence"
            else BetweenBenignSamples
        ),
    )
    results.append(result["results"])

display(result["availability"])
display(pd.concat(results, ignore_index=True))

,Attack,AvailableAttackFlows,OriginalStart,OriginalEnd
0,SSDP,2610611,2018-12-01 12:23:13.663477,2018-12-01 12:36:57.627790
1,SYN,1582289,2018-12-01 13:30:30.741566,2018-12-01 13:34:27.403304
2,UDP,3134645,2018-12-01 12:36:57.628218,2018-12-01 13:04:45.928511


,Scenario,Family,BlockInstances,RepeatAttackBlocks,Status,Reason,Path,BlocksPath,Instances,AttackInstances
0,Consistency,FLOW-UDP-SYN-SSDP,25,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Consistency\Co...,data\scenario\FLOW-UDP-SYN-SSDP\Consistency\Co...,15075,75
1,Consistency,FLOW-UDP-SYN-SSDP,200,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Consistency\Co...,data\scenario\FLOW-UDP-SYN-SSDP\Consistency\Co...,15600,600
2,Consistency,FLOW-UDP-SYN-SSDP,1000,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Consistency\Co...,data\scenario\FLOW-UDP-SYN-SSDP\Consistency\Co...,18000,3000
3,Generalization,FLOW-UDP-SYN-SSDP,25,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Generalization...,data\scenario\FLOW-UDP-SYN-SSDP\Generalization...,15075,75
4,Generalization,FLOW-UDP-SYN-SSDP,200,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Generalization...,data\scenario\FLOW-UDP-SYN-SSDP\Generalization...,15600,600
5,Generalization,FLOW-UDP-SYN-SSDP,1000,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Generalization...,data\scenario\FLOW-UDP-SYN-SSDP\Generalization...,18006,3000
6,Adaptation,FLOW-UDP-SYN-SSDP,25,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Adaptation\Ada...,data\scenario\FLOW-UDP-SYN-SSDP\Adaptation\Ada...,15077,75
7,Adaptation,FLOW-UDP-SYN-SSDP,200,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Adaptation\Ada...,data\scenario\FLOW-UDP-SYN-SSDP\Adaptation\Ada...,15602,600
8,Adaptation,FLOW-UDP-SYN-SSDP,1000,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Adaptation\Ada...,data\scenario\FLOW-UDP-SYN-SSDP\Adaptation\Ada...,18002,3000
9,Recurrence,FLOW-UDP-SYN-SSDP,25,False,Gerado,,data\scenario\FLOW-UDP-SYN-SSDP\Recurrence\Rec...,data\scenario\FLOW-UDP-SYN-SSDP\Recurrence\Rec...,15152,150
